In [1]:
from pathlib import Path
import os
import pandas as pd
import random
#PATHING SCRIPT FOR EVERY EXCERSISE - PROJECT - WORKSPACE
# 1.Literal definition of route pathings(Every user of remote repository must config this pathing in order to find the local repository of his computer)
# My case:
ROOT = Path("/home/josu/Documentos/DataScienceCourse")

# We verify the existence before continue
if not ROOT.exists():
    raise FileNotFoundError(f"❌ The route {ROOT} doesn't exist. Check it.")

# 2. Fix the workspace
os.chdir(ROOT)
print(f"✅ Worskspace enabled!: {os.getcwd()}")

# 3. Define relative pathings to work properly
DATA_TABLES = ROOT / "data" / "raw" / "Tables"

# Let's verify our table folder:
if not DATA_TABLES.exists():
    # If it fails, monitorize the issue: 
    data_dir = ROOT / "data"
    if data_dir.exists():
        print(f"⚠️ the folder 'data' exists but it doesn't have any 'Tables'. 'data' content: {os.listdir(data_dir)}")
    else:
        print(f"⚠️ The folder 'data' doesn't exist on ROOT workspace: {os.listdir(ROOT)}")
    raise FileNotFoundError(f"❌ The tables route {DATA_TABLES} is missing.")

print(f"📂 Tables route detected: {DATA_TABLES}")


✅ Worskspace enabled!: /home/josu/Documentos/DataScienceCourse
📂 Tables route detected: /home/josu/Documentos/DataScienceCourse/data/raw/Tables


1.A) Target Data Source: Country Name / ISO code all countries csv.

# ==========================================================
# UNIVERSAL HTML TABLE SCRAPER
# ----------------------------------------------------------
# Purpose:
#     Download a rendered webpage and extract a table
#     identified by its column headers.
#
# Workflow:
#     URL
#        ↓
#     Playwright
#        ↓
#     HTML
#        ↓
#     BeautifulSoup
#        ↓
#     Find matching table
#        ↓
#     pandas.DataFrame
#
# Returns:
#     pandas.DataFrame
# ==========================================================

from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
from io import StringIO
import pandas as pd


async def scrape_table(
    url: str,
    expected_columns: list[str],
    headless: bool = True,
    executable_path: str = "/usr/bin/chromium",
    wait_until: str = "domcontentloaded"
) -> pd.DataFrame:
    """
    Download a webpage and extract a table using its header names.

    Parameters
    ----------
    url : str
        Target webpage URL.

    expected_columns : list[str]
        Column names used to identify the correct table.

    headless : bool
        Launch Chromium without GUI.

    executable_path : str
        Local Chromium executable.

    wait_until : str
        Page loading strategy.
        Options:
            "load"
            "domcontentloaded"
            "networkidle"

    Returns
    -------
    pandas.DataFrame
        Extracted table.
    """

    # ------------------------------------------
    # Download rendered HTML
    # ------------------------------------------

    pw = await async_playwright().start()

    browser = await pw.chromium.launch(
        executable_path=executable_path,
        headless=headless
    )

    page = await browser.new_page()

    await page.goto(
        url,
        wait_until=wait_until
    )

    html = await page.content()

    await browser.close()
    await pw.stop()

    # ------------------------------------------
    # Search every HTML table
    # ------------------------------------------

    soup = BeautifulSoup(html, "lxml")

    tables = soup.find_all("table")

    # ------------------------------------------
    # Find matching table
    # ------------------------------------------

    for table in tables:

        header = table.find("tr")

        if header is None:
            continue

        cells = header.find_all(["th", "td"])

        columns = [
            cell.get_text(strip=True)
            for cell in cells
        ]

        if all(col in columns for col in expected_columns):

            df = pd.read_html(
                StringIO(str(table))
            )[0]

            return df

    raise ValueError(
        "No table matching the expected columns was found."
    )

In [ ]:
# ==========================================================
# UNIVERSAL HTML TABLE SCRAPER
# ----------------------------------------------------------
# Purpose:
#     Download a rendered webpage and extract a table
#     identified by its column headers.
#
# Matching strategy:
#     Header names DO NOT need an exact match.
#     The expected header only needs to be contained
#     inside the real HTML header.
#
# Example:
#
#     HTML Header:
#         "⇅ Official language(s)"
#
#     Expected:
#         "Official language(s)"
#
#     → MATCH
#
# Returns:
#     pandas.DataFrame
# ==========================================================

from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
from io import StringIO
import pandas as pd


async def scrape_table(
    url: str,
    expected_columns: list[str],
    headless: bool = True,
    executable_path: str = "/usr/bin/chromium",
    wait_until: str = "domcontentloaded"
) -> pd.DataFrame:

    # ------------------------------------------------------
    # Download HTML
    # ------------------------------------------------------

    pw = await async_playwright().start()

    browser = await pw.chromium.launch(
        executable_path=executable_path,
        headless=headless
    )

    page = await browser.new_page()

    await page.goto(
        url,
        wait_until=wait_until
    )

    html = await page.content()

    await browser.close()
    await pw.stop()

    # ------------------------------------------------------
    # Parse HTML
    # ------------------------------------------------------

    soup = BeautifulSoup(html, "lxml")

    tables = soup.find_all("table")

    # ------------------------------------------------------
    # Search matching table
    # ------------------------------------------------------

    for table in tables:

        header = table.find("tr")

        if header is None:
            continue

        cells = header.find_all(["th", "td"])

        html_headers = [
            cell.get_text(" ", strip=True)
            for cell in cells
        ]

        # --------------------------------------------------
        # Check if every expected column exists
        # (partial matching)
        # --------------------------------------------------

        valid = True

        for expected in expected_columns:

            if not any(
                expected.lower() in header.lower()
                for header in html_headers
            ):
                valid = False
                break

        if not valid:
            continue

        # --------------------------------------------------
        # Convert HTML table to DataFrame
        # --------------------------------------------------

        df = pd.read_html(
            StringIO(str(table))
        )[0]

        # --------------------------------------------------
        # Keep only requested columns
        # --------------------------------------------------

        selected_columns = {}

        for expected in expected_columns:

            for real in df.columns:

                if expected.lower() in str(real).lower():

                    selected_columns[expected] = real
                    break

        df = df[list(selected_columns.values())]

        # Rename columns using EXPECTED_COLUMNS names

        df.columns = list(selected_columns.keys())

        return df

    raise ValueError(
        "No matching table was found."
    )

In [ ]:
# ==========================================================
# Block 2 - User Configuration
# SCRAPER CONFIGURATION
# ==========================================================

URL = "https://es.wikipedia.org/wiki/ISO_3166-1_alfa-2"

EXPECTED_COLUMNS = [

    "Código",
    "Nombre del país",
    "Año",
    "ccTLD",
    "ISO 3166-2",
    "Notas"

]

OUTPUT_FILE = DATA_TABLES / "CountryNames.csv"

In [ ]:
# ==========================================================
# Block 3 - Execute Scraper
# RUN SCRAPER
# ==========================================================

df = await scrape_table(
    url=URL,
    expected_columns=EXPECTED_COLUMNS
)

display(df.head())

print(f"\nRows: {len(df)}")
print(f"Columns: {len(df.columns)}")

In [ ]:
# ==========================================================
# Block 4 - Save CSV
# SAVE DATAFRAME
# ==========================================================

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8"
)

print(f"CSV successfully saved:\n{OUTPUT_FILE}")

1.B) Target Data Source: Official Languages.

In [ ]:
# ==========================================================
# Block 2 - User Configuration
# SCRAPER CONFIGURATION
# ==========================================================

URL2 = "https://en.wikipedia.org/wiki/List_of_official_languages_by_country_and_territory"

EXPECTED_COLUMNS2 = [

    "Country/Region",
    "Number of official (including de facto)",
    "Official language(s)",
    "National language(s)",
    "Regional language(s)",
    "Minority language(s)",
    "Widely spoken"

]

OUTPUT_FILE = DATA_TABLES / "CountryLangs.csv"

Country/Region


In [ ]:
# ==========================================================
# Block 3 - Execute Scraper
# RUN SCRAPER
# ==========================================================

df2 = await scrape_table(
    url=URL2,
    expected_columns=EXPECTED_COLUMNS2
)

display(df2.head())

print(f"\nRows: {len(df2)}")
print(f"Columns: {len(df2.columns)}")